# LeXi v2: Adaptive Technical Vocabulary Tutor

`LeXi`는 영어 기술 문서를 읽는 한국인 개발자를 위한 학습 에이전트다.
이 노트북은 `challenge/12` README에 맞춰, 설계와 구현을 함께 진행하는 LeXi v2의 작업 공간이다.

이번 단계에서는 우선 LeXi v2의 상태 모델과 structured output 스키마를 정리한다.
            


In [ ]:
from __future__ import annotations

import os
import re
from datetime import datetime, timezone
import sqlite3
import tempfile
from pathlib import Path
from pprint import pprint
from typing import Literal

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from pydantic import BaseModel, Field
from typing_extensions import TypedDict

load_dotenv(Path.cwd() / ".env")

if not os.getenv("GOOGLE_API_KEY"):
    raise RuntimeError("GOOGLE_API_KEY must be set in the project .env file.")

llm = init_chat_model(
    model=os.getenv("GOOGLE_GENAI_MODEL", "gemini-3-flash-preview"),
    model_provider="google_genai",
    temperature=0,
)

UTC = timezone.utc


## Foundation Building

LeXi v2는 `MessagesState` 대신 `custom state`를 사용한다.
핵심은 대화 전체를 누적하는 것이 아니라, 입력을 분류하고 학습 카드와 복습 기록을 안정적으로 관리하는 것이다.
            


In [ ]:
RouteName = Literal["paragraph", "single_term", "review", "reentry", "invalid"]
ReviewResult = Literal["correct", "wrong", "unknown"]
StudyPriority = Literal["high", "medium", "low"]


class VocabularyEntry(TypedDict):
    word: str
    lemma: str
    meaning_in_context: str
    source_sentence: str
    context_note: str
    why_it_matters: str
    study_priority: StudyPriority


class MemoryRecord(TypedDict):
    word: str
    lemma: str
    meaning_in_context: str
    source_sentence: str
    context_note: str
    created_at: str
    review_count: int
    last_reviewed_at: str | None
    last_review_result: ReviewResult | None


class TermEvidence(TypedDict):
    term: str
    source_sentences: list[str]


class ReviewState(TypedDict):
    current_word: str
    current_source_sentence: str
    expected_meaning: str
    user_answer: str
    judgment: ReviewResult
    explanation: str


class LearningState(TypedDict):
    input_text: str
    route: RouteName | None
    candidate_words: list[str]
    terms_to_study: list[str]
    term_evidence: dict[str, TermEvidence]
    vocabulary_entries: list[VocabularyEntry]
    memory_records: list[MemoryRecord]
    review_queue: list[str]
    review_state: ReviewState | None
    review_history: list[dict[str, str]]
    continue_review: bool | None
    assistant_message: str | None
    error_message: str | None
    session_review_limit: int


class RouteDecision(BaseModel):
    route: RouteName = Field(description="Detected learning route for the current input")
    reason: str = Field(description="Short explanation for why the route was chosen")


class CandidateWords(BaseModel):
    candidate_words: list[str] = Field(
        description="Important English technical words or short terms worth studying. Maximum 5 items."
    )


class NormalizedTerms(BaseModel):
    terms_to_study: list[str] = Field(
        description="Normalized English terms to study. Maximum 5 items."
    )


class VocabularyEntryModel(BaseModel):
    word: str = Field(description="Word or phrase as it appears in the text")
    lemma: str = Field(description="Normalized lemma")
    meaning_in_context: str = Field(description="Korean meaning in the current technical context")
    source_sentence: str = Field(description="Source sentence from the original English text")
    context_note: str = Field(description="Short Korean explanation of why this meaning fits the context")
    why_it_matters: str = Field(description="Short Korean explanation of why this term matters for technical reading")
    study_priority: StudyPriority = Field(description="Study priority level for the learner")


class VocabularyEntries(BaseModel):
    vocabulary_entries: list[VocabularyEntryModel]


class ReviewJudgment(BaseModel):
    judgment: ReviewResult = Field(description="Whether the user's Korean answer is correct, wrong, or unknown")
    explanation: str = Field(description="Short Korean explanation used when the user needs feedback")


route_llm = llm.with_structured_output(RouteDecision)
candidate_llm = llm.with_structured_output(CandidateWords)
term_normalizer_llm = llm.with_structured_output(NormalizedTerms)
entry_llm = llm.with_structured_output(VocabularyEntries)
review_judge_llm = llm.with_structured_output(ReviewJudgment)


def make_initial_state(input_text: str = "") -> LearningState:
    return {
        "input_text": input_text,
        "route": None,
        "candidate_words": [],
        "terms_to_study": [],
        "term_evidence": {},
        "vocabulary_entries": [],
        "memory_records": [],
        "review_queue": [],
        "review_state": None,
        "review_history": [],
        "continue_review": None,
        "assistant_message": None,
        "error_message": None,
        "session_review_limit": 3,
    }
            


In [ ]:
sample_state = make_initial_state("Caching reduces latency, but stale data can break correctness.")
pprint(sample_state)
print("schema_ready_at", datetime.now(UTC).isoformat())
            


## Step 2. Request Routing

LeXi v2는 입력을 `paragraph`, `single_term`, `review`, `reentry`, `invalid`로 먼저 분류한다.
이 단계는 Conditional Edge의 기준이 되므로, 모호함보다 예측 가능성을 우선한다.
            


In [ ]:
REVIEW_KEYWORDS = {
    "review",
    "quiz",
    "review words",
    "review vocabulary",
    "복습",
    "퀴즈",
    "다시 보기",
    "단어 복습",
}

ENGLISH_TOKEN_RE = re.compile(r"[A-Za-z][A-Za-z0-9_-]*")
HANGUL_RE = re.compile(r"[가-힣]")


def contains_hangul(text: str) -> bool:
    return bool(HANGUL_RE.search(text))


def english_tokens(text: str) -> list[str]:
    return ENGLISH_TOKEN_RE.findall(text)


def is_explicit_review_request(text: str) -> bool:
    normalized = re.sub(r"\s+", " ", text.strip().lower())
    if normalized in REVIEW_KEYWORDS:
        return True
    return any(keyword in normalized for keyword in REVIEW_KEYWORDS)


def looks_like_single_term(text: str) -> bool:
    normalized = text.strip()
    if not normalized or contains_hangul(normalized):
        return False
    tokens = english_tokens(normalized)
    if not tokens or len(tokens) > 5:
        return False
    allowed = re.sub(r"[A-Za-z0-9_\-\s/().]", "", normalized)
    return not allowed and len(normalized) <= 80


def looks_like_english_paragraph(text: str) -> bool:
    normalized = text.strip()
    tokens = english_tokens(normalized)
    if len(tokens) < 6:
        return False
    if contains_hangul(normalized):
        return False
    sentence_markers = sum(normalized.count(mark) for mark in ".,;:!?")
    return sentence_markers > 0 or len(normalized.split()) >= 10


def analyze_request(state: LearningState) -> dict:
    text = state["input_text"].strip()
    if not text:
        return {
            "route": "reentry",
            "error_message": "입력이 비어 있습니다. 영어 기술 문장, 영어 용어, 또는 복습 요청을 입력해 주세요.",
        }

    if len(text) > 2000:
        return {
            "route": "reentry",
            "error_message": "입력이 2000자를 초과했습니다. 더 짧은 영어 기술 문장 또는 용어로 다시 입력해 주세요.",
        }

    if is_explicit_review_request(text):
        return {"route": "review", "error_message": None}

    if looks_like_single_term(text):
        return {"route": "single_term", "error_message": None}

    if looks_like_english_paragraph(text):
        return {"route": "paragraph", "error_message": None}

    return {
        "route": "invalid",
        "error_message": "LeXi는 영어 기술 문서 학습과 복습 요청만 처리합니다. 영어 기술 문장, 영어 용어, 또는 복습 요청을 입력해 주세요.",
    }
            


In [ ]:
routing_examples = [
    "review",
    "복습",
    "latency",
    "Caching reduces latency, but inconsistent invalidation can cause stale data.",
    "안녕하세요",
    "x" * 2001,
]

for example in routing_examples:
    result = analyze_request(make_initial_state(example))
    print(example[:60], "->", result["route"], "//", result["error_message"])
            


## Step 3. Learning Entry Paths

학습 모드는 두 갈래로 시작한다.
`paragraph`는 후보 단어를 뽑고, `single_term`은 입력을 정규화한 뒤, 둘 다 `terms_to_study`로 수렴한다.
            


In [ ]:
def dedupe_preserve_order(items: list[str]) -> list[str]:
    seen: set[str] = set()
    ordered: list[str] = []
    for item in items:
        normalized = item.strip()
        key = normalized.lower()
        if not normalized or key in seen:
            continue
        seen.add(key)
        ordered.append(normalized)
    return ordered


def extract_candidates(state: LearningState) -> dict:
    text = state["input_text"].strip()
    if state.get("route") != "paragraph" or not text:
        return {"candidate_words": [], "terms_to_study": []}

    prompt = f"""
You are helping a Korean developer study English technical writing.
Read the text and choose up to 5 English words or short terms that are worth learning.
Pick items that are important for understanding the technical meaning, not just rare words.
Return only the structured output.

Text:
{text}
""".strip()

    result = candidate_llm.invoke(prompt)
    candidate_words = dedupe_preserve_order(result.candidate_words)[:5]
    return {
        "candidate_words": candidate_words,
        "terms_to_study": candidate_words,
        "error_message": None,
    }


def normalize_term_request(state: LearningState) -> dict:
    text = state["input_text"].strip()
    if state.get("route") != "single_term" or not text:
        return {"terms_to_study": []}

    prompt = f"""
You are preparing study targets for a Korean developer reading English technical documents.
Normalize the input into up to 5 English technical terms to study.
Keep the terms concise and preserve the original technical meaning.
Return only the structured output.

Input:
{text}
""".strip()

    result = term_normalizer_llm.invoke(prompt)
    terms_to_study = dedupe_preserve_order(result.terms_to_study)[:5]
    return {
        "candidate_words": terms_to_study,
        "terms_to_study": terms_to_study,
        "error_message": None,
    }
            


In [ ]:
class FakeCandidateLLM:
    def invoke(self, prompt: str) -> CandidateWords:
        return CandidateWords(candidate_words=["latency", "invalidation", "latency"])


class FakeTermNormalizerLLM:
    def invoke(self, prompt: str) -> NormalizedTerms:
        return NormalizedTerms(terms_to_study=["backward compatibility", "schema"])


candidate_llm_backup = candidate_llm
term_normalizer_llm_backup = term_normalizer_llm
candidate_llm = FakeCandidateLLM()
term_normalizer_llm = FakeTermNormalizerLLM()

paragraph_state = make_initial_state(
    "Caching reduces latency, but inconsistent invalidation can cause stale data."
)
paragraph_state["route"] = "paragraph"
term_state = make_initial_state("backward compatibility")
term_state["route"] = "single_term"

print(extract_candidates(paragraph_state))
print(normalize_term_request(term_state))

candidate_llm = candidate_llm_backup
term_normalizer_llm = term_normalizer_llm_backup
            


## Step 4. Source Evidence Tool

LeXi v2는 모델이 문장을 지어내지 않도록, 원문에서 단어가 실제로 쓰인 문장을 찾는 custom tool을 사용한다.
이 단계에서는 해당 tool과 이를 사용하는 `enrich_term` node를 구현한다.
            


In [ ]:
SENTENCE_SPLIT_RE = re.compile(r"(?<=[.!?])\s+")


def split_sentences(text: str) -> list[str]:
    normalized = text.strip()
    if not normalized:
        return []
    if not SENTENCE_SPLIT_RE.search(normalized):
        return [normalized]
    return [sentence.strip() for sentence in SENTENCE_SPLIT_RE.split(normalized) if sentence.strip()]


@tool
def locate_source_sentences(term: str, input_text: str) -> list[str]:
    """Find source sentences from the original input that contain the given English term."""
    normalized_term = term.strip().lower()
    if not normalized_term:
        return []

    matched_sentences: list[str] = []
    for sentence in split_sentences(input_text):
        if normalized_term in sentence.lower():
            matched_sentences.append(sentence)

    return matched_sentences


def enrich_term(state: LearningState) -> dict:
    text = state["input_text"].strip()
    terms_to_study = state.get("terms_to_study", [])
    if not text or not terms_to_study:
        return {"term_evidence": {}}

    term_evidence: dict[str, TermEvidence] = {}
    for term in terms_to_study:
        source_sentences = locate_source_sentences.invoke({"term": term, "input_text": text})
        term_evidence[term] = {
            "term": term,
            "source_sentences": source_sentences,
        }

    return {"term_evidence": term_evidence, "error_message": None}
            


In [ ]:
source_text = (
    "Caching reduces latency, but inconsistent invalidation can cause stale data. "
    "When the API serializes responses, the schema must remain stable for backward compatibility."
)
source_state = make_initial_state(source_text)
source_state["terms_to_study"] = ["latency", "schema", "cache miss"]

print(locate_source_sentences.invoke({"term": "schema", "input_text": source_text}))
pprint(enrich_term(source_state))
            


## Step 5. Vocabulary Entry Builder

후보 단어와 근거 문장이 준비되면, LeXi v2는 이를 한국어 학습 카드로 정리한다.
이 결과는 사용자에게 보여주는 최종 학습 카드이자, 이후 memory에 저장될 기반 데이터다.
            


In [ ]:
def build_vocabulary_entries(state: LearningState) -> dict:
    text = state["input_text"].strip()
    terms_to_study = state.get("terms_to_study", [])
    term_evidence = state.get("term_evidence", {})
    if not text or not terms_to_study:
        return {"vocabulary_entries": []}

    evidence_blocks: list[str] = []
    for term in terms_to_study:
        evidence = term_evidence.get(term, {"term": term, "source_sentences": []})
        source_sentences = evidence.get("source_sentences", []) or ["(no exact source sentence found)"]
        evidence_blocks.append(
            f"Term: {term}\nSource sentences:\n- " + "\n- ".join(source_sentences)
        )

    prompt = f"""
You are building study cards for a Korean developer reading English technical documents.
Use the original text and the term evidence below.
For each study term, return:
- word as it appears in the text or the best matching surface form
- lemma
- Korean meaning in this technical context
- source sentence from the original text
- a short Korean context note
- a short Korean explanation of why the term matters
- study priority as high, medium, or low
Return only the structured output.

Original text:
{text}

Term evidence:
{chr(10).join(evidence_blocks)}
""".strip()

    result = entry_llm.invoke(prompt)
    vocabulary_entries = [entry.model_dump() for entry in result.vocabulary_entries]
    return {"vocabulary_entries": vocabulary_entries, "error_message": None}


In [ ]:
class FakeEntryLLM:
    def invoke(self, prompt: str) -> VocabularyEntries:
        return VocabularyEntries(
            vocabulary_entries=[
                VocabularyEntryModel(
                    word="latency",
                    lemma="latency",
                    meaning_in_context="지연 시간",
                    source_sentence="Caching reduces latency, but inconsistent invalidation can cause stale data.",
                    context_note="성능을 설명할 때 응답이 느려지는 시간을 뜻한다.",
                    why_it_matters="기술 문서에서 성능 병목을 이해할 때 자주 등장한다.",
                    study_priority="high",
                )
            ]
        )


entry_llm_backup = entry_llm
entry_llm = FakeEntryLLM()

entry_state = make_initial_state(
    "Caching reduces latency, but inconsistent invalidation can cause stale data."
)
entry_state["terms_to_study"] = ["latency"]
entry_state["term_evidence"] = {
    "latency": {
        "term": "latency",
        "source_sentences": [
            "Caching reduces latency, but inconsistent invalidation can cause stale data."
        ],
    }
}

pprint(build_vocabulary_entries(entry_state))

entry_llm = entry_llm_backup
            


## Step 6. SQLite Memory

LeXi v2는 학습 카드와 복습 결과를 SQLite에 저장한다.
이 단계에서는 메모리 스키마를 만들고, `save_memory` node가 학습 카드를 저장하도록 구현한다.
            


In [ ]:
MEMORY_DB_PATH = Path.cwd() / "challenge/12/lexi_memory.db"


def initialize_memory_db(db_path: Path = MEMORY_DB_PATH) -> None:
    db_path.parent.mkdir(parents=True, exist_ok=True)
    with sqlite3.connect(db_path) as conn:
        conn.execute(
            """
            CREATE TABLE IF NOT EXISTS study_memory (
                word TEXT PRIMARY KEY,
                lemma TEXT NOT NULL,
                meaning_in_context TEXT NOT NULL,
                source_sentence TEXT NOT NULL,
                context_note TEXT NOT NULL,
                created_at TEXT NOT NULL,
                review_count INTEGER NOT NULL DEFAULT 0,
                last_reviewed_at TEXT,
                last_review_result TEXT
            )
            """
        )
        conn.commit()


def load_memory_records(db_path: Path = MEMORY_DB_PATH) -> list[MemoryRecord]:
    if not db_path.exists():
        return []

    with sqlite3.connect(db_path) as conn:
        conn.row_factory = sqlite3.Row
        rows = conn.execute(
            """
            SELECT
                word,
                lemma,
                meaning_in_context,
                source_sentence,
                context_note,
                created_at,
                review_count,
                last_reviewed_at,
                last_review_result
            FROM study_memory
            ORDER BY created_at ASC
            """
        ).fetchall()

    return [dict(row) for row in rows]


def upsert_memory_record(entry: VocabularyEntry, db_path: Path = MEMORY_DB_PATH) -> None:
    initialize_memory_db(db_path)
    with sqlite3.connect(db_path) as conn:
        existing = conn.execute(
            "SELECT created_at, review_count, last_reviewed_at, last_review_result FROM study_memory WHERE word = ?",
            (entry["word"],),
        ).fetchone()

        created_at = existing[0] if existing else datetime.now(UTC).isoformat()
        review_count = existing[1] if existing else 0
        last_reviewed_at = existing[2] if existing else None
        last_review_result = existing[3] if existing else None

        conn.execute(
            """
            INSERT INTO study_memory (
                word,
                lemma,
                meaning_in_context,
                source_sentence,
                context_note,
                created_at,
                review_count,
                last_reviewed_at,
                last_review_result
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
            ON CONFLICT(word) DO UPDATE SET
                lemma = excluded.lemma,
                meaning_in_context = excluded.meaning_in_context,
                source_sentence = excluded.source_sentence,
                context_note = excluded.context_note,
                created_at = excluded.created_at,
                review_count = excluded.review_count,
                last_reviewed_at = excluded.last_reviewed_at,
                last_review_result = excluded.last_review_result
            """,
            (
                entry["word"],
                entry["lemma"],
                entry["meaning_in_context"],
                entry["source_sentence"],
                entry["context_note"],
                created_at,
                review_count,
                last_reviewed_at,
                last_review_result,
            ),
        )
        conn.commit()


def save_memory(state: LearningState, db_path: Path = MEMORY_DB_PATH) -> dict:
    vocabulary_entries = state.get("vocabulary_entries", [])
    if not vocabulary_entries:
        return {"memory_records": load_memory_records(db_path)}

    for entry in vocabulary_entries:
        upsert_memory_record(entry, db_path=db_path)

    return {"memory_records": load_memory_records(db_path), "error_message": None}
            


In [ ]:
with tempfile.TemporaryDirectory() as temp_dir:
    temp_db = Path(temp_dir) / "lexi_memory.db"
    temp_state = make_initial_state()
    temp_state["vocabulary_entries"] = [
        {
            "word": "latency",
            "lemma": "latency",
            "meaning_in_context": "지연 시간",
            "source_sentence": "Caching reduces latency.",
            "context_note": "응답 속도와 관련된 문맥이다.",
            "why_it_matters": "성능 관련 문서에서 중요하다.",
            "study_priority": "high",
        }
    ]

    saved = save_memory(temp_state, db_path=temp_db)
    pprint(saved)
            


## Step 7. Review Candidate Selection

review는 memory가 있어야 시작할 수 있다.
이 단계에서는 저장된 단어들 중에서 균형 있게 review 문제를 고르는 tool과, 이를 사용하는 `load_review_memory` node를 구현한다.
            


In [ ]:
def parse_reviewed_at(value: str | None) -> float:
    if not value:
        return float("-inf")
    try:
        return datetime.fromisoformat(value).timestamp()
    except ValueError:
        return float("-inf")


@tool
def select_review_candidates(
    memory_records: list[MemoryRecord],
    limit: int = 3,
    exclude_words: list[str] | None = None,
) -> list[str]:
    """Select balanced review candidates using review count, recency, and previous mistakes."""
    excluded = {word.lower() for word in (exclude_words or [])}
    eligible_records = [
        record for record in memory_records if record["word"].lower() not in excluded
    ]

    def sort_key(record: MemoryRecord) -> tuple[int, int, float, str]:
        wrong_priority = 0 if record.get("last_review_result") == "wrong" else 1
        return (
            record.get("review_count", 0),
            wrong_priority,
            parse_reviewed_at(record.get("last_reviewed_at")),
            record["word"].lower(),
        )

    ordered_records = sorted(eligible_records, key=sort_key)
    return [record["word"] for record in ordered_records[:limit]]


def load_review_memory(state: LearningState, db_path: Path = MEMORY_DB_PATH) -> dict:
    memory_records = load_memory_records(db_path)
    if not memory_records:
        return {
            "memory_records": [],
            "review_queue": [],
            "error_message": "아직 복습할 단어가 없습니다. 먼저 영어 기술 문장을 학습해 단어장을 만들어 주세요.",
        }

    review_queue = select_review_candidates.invoke(
        {
            "memory_records": memory_records,
            "limit": state.get("session_review_limit", 3),
            "exclude_words": [item["word"] for item in state.get("review_history", [])],
        }
    )
    return {
        "memory_records": memory_records,
        "review_queue": review_queue,
        "error_message": None,
    }
            


In [ ]:
with tempfile.TemporaryDirectory() as temp_dir:
    temp_db = Path(temp_dir) / "lexi_memory.db"
    initialize_memory_db(temp_db)

    sample_records = [
        {
            "word": "latency",
            "lemma": "latency",
            "meaning_in_context": "지연 시간",
            "source_sentence": "Latency affects response time.",
            "context_note": "성능 문맥",
            "why_it_matters": "중요 성능 용어",
            "study_priority": "high",
        },
        {
            "word": "schema",
            "lemma": "schema",
            "meaning_in_context": "스키마",
            "source_sentence": "The schema must remain stable.",
            "context_note": "데이터 구조 문맥",
            "why_it_matters": "API 문서에서 자주 보임",
            "study_priority": "medium",
        },
        {
            "word": "serialization",
            "lemma": "serialization",
            "meaning_in_context": "직렬화",
            "source_sentence": "Serialization changes the payload shape.",
            "context_note": "데이터 변환 문맥",
            "why_it_matters": "네트워크 전송 설명에 자주 등장",
            "study_priority": "medium",
        },
    ]

    for entry in sample_records:
        upsert_memory_record(entry, temp_db)

    with sqlite3.connect(temp_db) as conn:
        conn.execute(
            "UPDATE study_memory SET review_count = 2, last_reviewed_at = ?, last_review_result = 'correct' WHERE word = 'latency'",
            (datetime(2026, 3, 28, tzinfo=UTC).isoformat(),),
        )
        conn.execute(
            "UPDATE study_memory SET review_count = 1, last_reviewed_at = ?, last_review_result = 'wrong' WHERE word = 'schema'",
            (datetime(2026, 3, 20, tzinfo=UTC).isoformat(),),
        )
        conn.commit()

    review_state = make_initial_state("review")
    pprint(load_review_memory(review_state, db_path=temp_db))
            


## Step 8. Interactive Review Judging

review는 단어와 예문을 보여준 뒤 사용자가 한국어 뜻을 입력하는 상호작용으로 진행된다.
이 단계에서는 질문을 준비하고, hybrid 방식으로 정답을 판정하는 로직을 구현한다.
            


In [ ]:
def present_review_question(state: LearningState) -> dict:
    review_queue = state.get("review_queue", [])
    memory_records = state.get("memory_records", [])
    if not review_queue:
        return {
            "review_state": None,
            "error_message": "현재 review queue가 비어 있습니다.",
        }

    current_word = review_queue[0]
    current_record = next(
        (record for record in memory_records if record["word"] == current_word),
        None,
    )
    if not current_record:
        return {
            "review_state": None,
            "error_message": f"'{current_word}' 단어의 memory record를 찾지 못했습니다.",
        }

    review_state: ReviewState = {
        "current_word": current_record["word"],
        "current_source_sentence": current_record["source_sentence"],
        "expected_meaning": current_record["meaning_in_context"],
        "user_answer": "",
        "judgment": "unknown",
        "explanation": "",
    }
    return {"review_state": review_state, "error_message": None}


def normalize_korean_meaning(text: str) -> str:
    lowered = text.strip().lower()
    return re.sub(r"[^가-힣a-z0-9]", "", lowered)


def judge_review_answer(state: LearningState) -> dict:
    review_state = state.get("review_state")
    if not review_state:
        return {"error_message": "판정할 review 상태가 없습니다."}

    expected_meaning = review_state["expected_meaning"].strip()
    user_answer = review_state["user_answer"].strip()
    normalized_expected = normalize_korean_meaning(expected_meaning)
    normalized_answer = normalize_korean_meaning(user_answer)

    if not normalized_answer:
        updated_state = {
            **review_state,
            "judgment": "wrong",
            "explanation": f"정답은 '{expected_meaning}'입니다. 답안을 입력하지 않아 오답으로 처리했습니다.",
        }
        return {"review_state": updated_state, "error_message": None}

    if normalized_answer == normalized_expected or (
        normalized_answer and (normalized_answer in normalized_expected or normalized_expected in normalized_answer)
    ):
        updated_state = {
            **review_state,
            "judgment": "correct",
            "explanation": "정답입니다. 현재 문맥에서의 의미를 정확히 이해했습니다.",
        }
        return {"review_state": updated_state, "error_message": None}

    prompt = f"""
You are judging whether a Korean learner's answer matches the intended meaning of an English technical term.
Return only the structured output.
If the learner is wrong, explain briefly in Korean and include the correct Korean meaning.

English term: {review_state['current_word']}
Source sentence: {review_state['current_source_sentence']}
Expected Korean meaning: {expected_meaning}
User answer: {user_answer}
""".strip()

    llm_result = review_judge_llm.invoke(prompt)
    explanation = llm_result.explanation
    if llm_result.judgment == "wrong" and expected_meaning not in explanation:
        explanation = f"정답은 '{expected_meaning}'입니다. {explanation}"

    updated_state = {
        **review_state,
        "judgment": llm_result.judgment,
        "explanation": explanation,
    }
    return {"review_state": updated_state, "error_message": None}
            


In [ ]:
class FakeReviewJudgeLLM:
    def invoke(self, prompt: str) -> ReviewJudgment:
        return ReviewJudgment(
            judgment="wrong",
            explanation="정답은 '직렬화'입니다. 데이터 구조를 전송 가능한 형태로 바꾸는 의미입니다.",
        )


review_judge_llm_backup = review_judge_llm
review_judge_llm = FakeReviewJudgeLLM()

review_memory_records = [
    {
        "word": "serialization",
        "lemma": "serialization",
        "meaning_in_context": "직렬화",
        "source_sentence": "Serialization changes the payload shape.",
        "context_note": "데이터 변환 문맥",
        "created_at": datetime.now(UTC).isoformat(),
        "review_count": 0,
        "last_reviewed_at": None,
        "last_review_result": None,
    }
]

question_state = make_initial_state("review")
question_state["memory_records"] = review_memory_records
question_state["review_queue"] = ["serialization"]
question_payload = present_review_question(question_state)

wrong_answer_state = {
    **question_state,
    **question_payload,
}
wrong_answer_state["review_state"]["user_answer"] = "직렬 처리"

pprint(question_payload)
pprint(judge_review_answer(wrong_answer_state))

review_judge_llm = review_judge_llm_backup
            


## Step 9. Review Persistence And Flow Control

review 결과는 단순 출력으로 끝나지 않고 memory에 반영되어야 한다.
이 단계에서는 정답/오답 결과를 저장하고, 다음 단어로 갈지 종료할지 판단하는 흐름을 구현한다.
            


In [ ]:
def update_review_memory(state: LearningState, db_path: Path = MEMORY_DB_PATH) -> dict:
    review_state = state.get("review_state")
    if not review_state:
        return {"error_message": "업데이트할 review 상태가 없습니다."}

    current_word = review_state["current_word"]
    judgment = review_state["judgment"]
    reviewed_at = datetime.now(UTC).isoformat()

    initialize_memory_db(db_path)
    with sqlite3.connect(db_path) as conn:
        conn.execute(
            """
            UPDATE study_memory
            SET review_count = review_count + 1,
                last_reviewed_at = ?,
                last_review_result = ?
            WHERE word = ?
            """,
            (reviewed_at, judgment, current_word),
        )
        conn.commit()

    remaining_queue = state.get("review_queue", [])[1:]
    review_history = state.get("review_history", []) + [
        {
            "word": current_word,
            "judgment": judgment,
            "user_answer": review_state["user_answer"],
            "expected_meaning": review_state["expected_meaning"],
        }
    ]

    return {
        "memory_records": load_memory_records(db_path),
        "review_queue": remaining_queue,
        "review_history": review_history,
        "assistant_message": review_state["explanation"],
        "error_message": None,
    }


def next_review_or_finish(state: LearningState) -> dict:
    review_state = state.get("review_state")
    review_history = state.get("review_history", [])
    review_queue = state.get("review_queue", [])
    continue_review = state.get("continue_review")
    session_limit = state.get("session_review_limit", 3)

    if not review_state:
        return {"error_message": "다음 review 단계를 결정할 review 상태가 없습니다."}

    judgment = review_state["judgment"]
    completed_count = len(review_history)
    reached_limit = completed_count >= session_limit

    if judgment == "correct":
        if reached_limit:
            return {
                "assistant_message": "기본 3개 단어 복습을 마쳤습니다.",
                "continue_review": False,
                "review_state": None,
                "error_message": None,
            }
        if continue_review is None:
            return {
                "assistant_message": "정답입니다. 복습을 계속할까요?",
                "review_state": review_state,
                "error_message": None,
            }
        if continue_review is False:
            return {
                "assistant_message": "이번 복습을 여기서 마칩니다.",
                "review_state": None,
                "error_message": None,
            }
        if review_queue:
            return {
                "assistant_message": "좋습니다. 다음 단어로 넘어갑니다.",
                "review_state": None,
                "error_message": None,
            }
        return {
            "assistant_message": "남은 복습 단어가 없습니다.",
            "review_state": None,
            "error_message": None,
        }

    if reached_limit or not review_queue:
        return {
            "assistant_message": f"{review_state['explanation']} 기본 3개 단어 복습을 마쳤습니다.",
            "review_state": None,
            "continue_review": False,
            "error_message": None,
        }

    return {
        "assistant_message": f"{review_state['explanation']} 다음 단어로 넘어갑니다.",
        "review_state": None,
        "continue_review": True,
        "error_message": None,
    }
            


In [ ]:
with tempfile.TemporaryDirectory() as temp_dir:
    temp_db = Path(temp_dir) / "lexi_memory.db"
    initialize_memory_db(temp_db)
    upsert_memory_record(
        {
            "word": "latency",
            "lemma": "latency",
            "meaning_in_context": "지연 시간",
            "source_sentence": "Caching reduces latency.",
            "context_note": "응답 속도 문맥",
            "why_it_matters": "성능 문서에서 중요함",
            "study_priority": "high",
        },
        db_path=temp_db,
    )

    review_state = {
        **make_initial_state("review"),
        "review_queue": ["latency", "schema"],
        "review_state": {
            "current_word": "latency",
            "current_source_sentence": "Caching reduces latency.",
            "expected_meaning": "지연 시간",
            "user_answer": "지연 시간",
            "judgment": "correct",
            "explanation": "정답입니다. 현재 문맥에서의 의미를 정확히 이해했습니다.",
        },
    }

    updated = update_review_memory(review_state, db_path=temp_db)
    continued = next_review_or_finish({**review_state, **updated})
    pprint(updated)
    pprint(continued)
            
